# Does LightGBM's cross-conformal really cost more wall-clock than TabPFN's?

This notebook settles **P5**, the one pre-registered prediction in
[tabpfn-conformal](https://github.com/ilyas-elm/tabpfn-conformal) that the project
still reports as unsettled.

**Why it was unsettled.** The project's main experiments measured TabPFN through
the Prior Labs API, running on their GPUs, against LightGBM running on a laptop
CPU. That is not a race: most of the TabPFN number was network round-trip. Every
wall-clock row in experiment E4 carries `wallclock_comparable: false` for exactly
that reason, and the repository declines to quote the figure rather than resting
a claim on it.

**What this does.** Both models, one machine, one accelerator, using TabPFN's
downloadable weights so that nothing crosses the network while the clock is
running. The E4 protocol is otherwise unchanged: matched label budgets, the same
pool and evaluation construction, 2 budgets x 3 seeds x 2 families x 2 strategies,
on Bank Account Fraud (Jesus et al., NeurIPS 2022).

**What it cannot do.** It measures *local* TabPFN, not the managed API, so these
timings do not reproduce the API numbers and are not meant to. The gradient-fit
count is the hardware-independent comparison and does not move either way: 0 for
TabPFN, 1 for LightGBM split, 6 for LightGBM cross at K=5.

## The hardware

The entire point of this run is that both models share one accelerator, so the
machine is recorded here rather than asserted. If this reports no GPU, the
analysis at the end says so and draws no conclusion.

In [ ]:
import subprocess, sys

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("gpu:   ", gpu or "NONE VISIBLE, the accelerator is not set")
print("python:", sys.version.split()[0])

## Credentials and data

TabPFN asks for a one-time licence acceptance before it will download weights for
local use, and outside an interactive terminal it reads an API key from the
environment instead of prompting. The key **authorises the download and nothing
else**: inference runs on the weights, on this machine, which is what keeps the
measurement clean. It is read from a Kaggle secret, so it is not stored in the
notebook.

The dataset is located by searching for the file, because a Kaggle input folder
is named after whichever copy of the dataset was attached.

In [ ]:
import os, pathlib
from kaggle_secrets import UserSecretsClient

for label in ("tabpfn v3.5", "TABPFN_TOKEN", "tabpfn_token", "conform"):
    try:
        os.environ["TABPFN_TOKEN"] = UserSecretsClient().get_secret(label)
        print("secret:", label)
        break
    except Exception:
        continue
else:
    raise SystemExit("No TabPFN secret found. See the appendix at the bottom.")

base = next(pathlib.Path("/kaggle/input").rglob("Base.csv"), None)
if base is None:
    attached = [p.name for p in pathlib.Path("/kaggle/input").glob("*")]
    raise SystemExit(f"Base.csv not found. Inputs attached: {attached or 'none'}. "
                     "See the appendix at the bottom.")
DATA = str(base.parent)
print("data:  ", DATA)

## The code being measured

Cloned from the repository rather than pasted into a cell, so this notebook cannot
drift from the library it claims to be measuring. `experiments/kaggle/wallclock.py`
is about 200 lines and readable there.

In [ ]:
%cd /kaggle/working
!pip install -q tabpfn lightgbm
!rm -rf /kaggle/working/tabpfn-conformal
!git clone -q https://github.com/ilyas-elm/tabpfn-conformal.git
%cd /kaggle/working/tabpfn-conformal
!pip install -q -e .

## The measurement

Both families are warmed up before anything is timed. TabPFN does not load its
weights until `fit`, and on a fresh machine that first call also *downloads* them,
876 MB here. Timing it would have charged TabPFN a large network cost in the one
experiment whose entire purpose is to have no network in it.

Results are written after every configuration rather than at the end, so an
interrupted session still leaves usable rows. Roughly half an hour on a T4.

In [ ]:
!python experiments/kaggle/wallclock.py --data "$DATA"

## The verdict

The two families are run on the same seeds, so the comparison is paired rather
than a difference of means, and it reports the standard error and the number of
seeds behind it. A gap smaller than two standard errors is reported as no
separation, not as a win.

In [ ]:
!python experiments/analyze_kaggle.py

## Reading this

Whatever the timings say, the claim the project leads with is the
hardware-independent one: **0 gradient-trained fits against LightGBM's 6** at K=5.
That is why K-fold cross-conformal is affordable on TabPFN at all, and no choice
of machine changes it.

Wall-clock is the secondary question. P5 predicted it would favour TabPFN. The
API-based measurement suggested the opposite but could not be trusted, for the
confound described at the top, and this run is what replaces it. P5 is one of
five pre-registered predictions in the project, four of which were falsified,
including two of the authors' own about cost.

Method, benchmarks and the full falsification table:
<https://github.com/ilyas-elm/tabpfn-conformal>

---

## Appendix: reproducing this

Copy & Edit runs this on Kaggle's hardware. Four things have to be set first, and
the notebook stops in its first seconds with a specific message if any is missing.

1. **Accelerator**: GPU T4 x2, under Session options. On CPU the run settles
   nothing and the analysis refuses to draw a conclusion.
2. **Internet**: on, under Session options. It is off by default, and the install
   and the clone both need it. Both this and the GPU require Kaggle phone
   verification.
3. **A TabPFN API key**, as a Kaggle secret attached to the notebook, from
   <https://ux.priorlabs.ai/account>, with the licence accepted on the Licenses
   tab of that site. The cell above tries a few likely secret labels and prints
   the one it found; a different label can be
   added to that list.
4. **The dataset**: Add Input, `Bank Account Fraud Dataset NeurIPS 2022`. The
   folder name does not matter, the notebook searches for the file.

Running it outside Kaggle needs the same two packages, the `TABPFN_TOKEN`
environment variable, and `--data` pointed at a directory holding `Base.csv`.